1. Environment Setup

This notebook prepares the environment for our Mini-BART research project.

In [ ]:
# %pip install -q "numpy==1.26.4" "transformers<5" datasets evaluate rouge_score sentencepiece accelerate

In [ ]:
# %pip install -q "scipy==1.15.3"

In [ ]:
# %pip install -q "numpy==2.2.6" "transformers==4.57.6" "datasets==3.6.0" evaluate rouge_score sentencepiece accelerate tensorboard pandas matplotlib pyyaml

2. Imports

Import the libraries required for this project.

In [ ]:
import numpy as np
import scipy
import torch
import transformers

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

3. GPU Verification

Verify that PyTorch can access our Colab GPU.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import torch
import numpy as np
import scipy
import transformers
import datasets
import evaluate
import huggingface_hub
import fsspec

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nNumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)
print("\nAll imports successful!")

4. Configuration

Define project-wide settings.

## 4. Loading the Pretrained BART Teacher

We use facebook/bart-large-cnn as our teacher model.

It is already pretrained and fine-tuned for CNN/DailyMail summarization.

Our objective is to use this teacher to guide the training of a smaller BART student through knowledge distillation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

teacher = teacher.to("cuda")
teacher.eval()

print("Teacher loaded successfully!")
print("Model:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in teacher.parameters()))

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create a permanent storage location
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
os.makedirs(TEACHER_DIR, exist_ok=True)

# Save the model and tokenizer
teacher.save_pretrained(TEACHER_DIR)
tokenizer.save_pretrained(TEACHER_DIR)

print("Teacher model and tokenizer saved successfully!")
print("Location:", TEACHER_DIR)


In [ ]:
print(os.listdir(TEACHER_DIR))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

teacher_path = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

test_tokenizer = AutoTokenizer.from_pretrained(teacher_path)
test_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_path)

print("Reload successful!")
print("Parameters:", sum(p.numel() for p in test_model.parameters()))

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

print("Teacher directory exists:", os.path.exists(TEACHER_DIR))
print("Files:", os.listdir(TEACHER_DIR))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    TEACHER_DIR
).to("cuda")

teacher.eval()

print("Teacher loaded successfully!")
print("Parameters:", sum(p.numel() for p in teacher.parameters()))
print("Device:", next(teacher.parameters()).device)

In [ ]:
article = """
The Indian Space Research Organisation (ISRO) successfully launched
its latest Earth observation satellite aboard a Polar Satellite Launch
Vehicle (PSLV) from the Satish Dhawan Space Centre in Sriharikota.
The satellite will provide high-resolution images to support
agriculture, disaster management, urban planning, and environmental
monitoring. Scientists stated that the mission demonstrates India's
continued progress in developing reliable space technology.
"""

inputs = tokenizer(
    article,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to("cuda")

with torch.no_grad():
    summary_ids = teacher.generate(
        **inputs,
        max_length=100,
        num_beams=4,
        forced_bos_token_id=0
    )

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Teacher summary:\n", summary)

In [ ]:

from datasets import load_dataset

# Load 5 examples from the test split
dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

print("Dataset loaded successfully!")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)


In [ ]:
import datasets
import huggingface_hub

print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="test[:5]"
)

print("Dataset loaded successfully!")
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)

In [ ]:
sample = dataset[0]

print("ARTICLE:\n")
print(sample["article"][:1500])

print("\nREFERENCE SUMMARY:\n")
print(sample["highlights"])

In [ ]:

import torch
teacher.eval()
teacher_results = []
for i, example in enumerate(dataset):

    article = example["article"]
    reference = example["highlights"]

    # Tokenize the article
    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to("cuda")

    # Generate summary
    with torch.no_grad():
        summary_ids = teacher.generate(
            **inputs,
            max_length=128,
            min_length=30,
            num_beams=4,
            forced_bos_token_id=0
        )

    # Convert generated tokens into text
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    teacher_results.append({
        "id": example["id"],
        "article": article,
        "reference_summary": reference,
        "teacher_summary": generated_summary
    })

    print(f"Completed article {i + 1}/5")

print("\nAll teacher summaries generated!")


In [ ]:
for i, result in enumerate(teacher_results):

    print(f"\n{'='*80}")
    print(f"ARTICLE {i+1}")
    print(f"{'='*80}")

    print("\nREFERENCE SUMMARY:\n")
    print(result["reference_summary"])

    print("\nTEACHER SUMMARY:\n")
    print(result["teacher_summary"])

In [ ]:
import json
import os

RESULTS_DIR = (
    "/content/drive/MyDrive/research_artifacts/mini_bart/"
    "teacher_baseline"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(
    RESULTS_DIR,
    "teacher_baseline_5_samples.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        teacher_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved to:", output_path)

In [ ]:
for i, result in enumerate(teacher_results):
    print(f"\nArticle {i+1}")
    print("ID:", result["id"])
    print("Article length:", len(result["article"]))
    print("Reference:", result["reference_summary"][:150])
    print("Teacher:", result["teacher_summary"])

In [ ]:
print("Total results:", len(teacher_results))

print("\nArticle 5")
print("ID:", teacher_results[4]["id"])
print("Article length:", len(teacher_results[4]["article"]))

print("\nReference summary:")
print(teacher_results[4]["reference_summary"])

print("\nTeacher summary:")
print(teacher_results[4]["teacher_summary"])

In [ ]:
%pip install -q evaluate rouge_score

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
print("ROUGE metric loaded successfully!")

In [ ]:
predictions = [
    result["teacher_summary"]
    for result in teacher_results
]

references = [
    result["reference_summary"]
    for result in teacher_results
]

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("Teacher ROUGE Scores:\n")

for metric, score in rouge_scores.items():
    print(f"{metric}: {score:.4f}")

MEASURING EXTRACTIVENESS (E0)

Unigram overlap: copied individual words

Bigram overlap: copied consecutive two-word phrases

4-gram overlap: copied consecutive four-word phrases

copy rate= summary n-grams found in source/total summary n grams

Novel rate = 1-copy rate

In [ ]:
import re
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())
def get_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]
def source_overlap(source, summary, n):
    source_tokens = tokenize(source)
    summary_tokens = tokenize(summary)

    source_ngrams = set(get_ngrams(source_tokens, n))
    summary_ngrams = get_ngrams(summary_tokens, n)

    if not summary_ngrams:
        return 0.0

    copied = sum(
        ngram in source_ngrams
        for ngram in summary_ngrams
    )

    return copied / len(summary_ngrams)

In [ ]:
copy_rates = {1: [], 2: [], 4: []}
for result in teacher_results:

    # Reconstruct the exact truncated article input
    encoded = tokenizer(
        result["article"],
        max_length=1024,
        truncation=True
    )

    source_text = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    summary = result["teacher_summary"]

    for n in [1, 2, 4]:
        rate = source_overlap(source_text, summary, n)
        copy_rates[n].append(rate)


print("Teacher Source Overlap:\n")

for n in [1, 2, 4]:
    avg_copy = sum(copy_rates[n]) / len(copy_rates[n])
    avg_novel = 1 - avg_copy

    print(f"{n}-gram Copy Rate:  {avg_copy:.4f}")
    print(f"{n}-gram Novel Rate: {avg_novel:.4f}\n")

SAVING BASELINE RESULT

will compare it with E1 AND E2 LATER

SAVING ROUGE METRICS and EXTRACTIVENESS RECORD IN GOOGLE DRIVE

In [ ]:
import json
import os
baseline_metrics = {
    "experiment": "E0_teacher",
    "num_examples": len(teacher_results),
    "rouge": rouge_scores,
    "source_overlap": {
        f"{n}-gram": {
            "copy_rate": sum(copy_rates[n]) / len(copy_rates[n]),
            "novel_rate": 1 - sum(copy_rates[n]) / len(copy_rates[n])
        }
        for n in [1, 2, 4]
    }
}

RESULTS_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher_baseline"
os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(RESULTS_DIR, "teacher_metrics.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=4)

print("Teacher baseline metrics saved!")
print(output_path)

In [ ]:
import os

print("Current working directory:", os.getcwd())

print("\nChecking project directory:")

for path in [
    "projects/01-mini-bart/src/metrics.py",
    "/content/Research/projects/01-mini-bart/src/metrics.py"
]:
    print(path, "->", os.path.exists(path))

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:20])

In [ ]:
import os
research_path = "/content/drive/MyDrive/Research"

print("Research folder exists:", os.path.exists(research_path))

if os.path.exists(research_path):
    print("\nContents:")
    print(os.listdir(research_path))

VS Code (Windows)
Source of truth, Write and edit metrics.py, notebooks and other project files.

Google Drive
Make the repository files accessible to Colab.

Colab T4
Import and execute the Python modules remotely.

cloning REPO IN DRIVE

In [ ]:
#cloning repo in drive
%cd /content/drive/MyDrive

!git clone https://github.com/HeavenlyDemon27/Research.git

verifying the cloning

In [ ]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/Research"

print("Repository exists:", os.path.exists(PROJECT_ROOT))
print("Project files:", os.listdir(PROJECT_ROOT))

Sync the latest code to the remote runtime

This updates the Drive copy with the metrics.py I just pushed.

In [ ]:
%cd /content/drive/MyDrive/Research
!git pull --ff-only origin main

Import our functions

We're importing our own reusable functions rather than keeping all evaluation logic inside the noteboo

In [ ]:
import sys
SRC_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart/src"

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from metrics import compute_rouge, compute_source_overlap

print("Metrics module imported successfully!")

Testing module against our teacher results

In [ ]:
predictions = [r["teacher_summary"] for r in teacher_results]
references = [r["reference_summary"] for r in teacher_results]
sources = []
for r in teacher_results:
    encoded = tokenizer(
        r["article"],
        max_length=1024,
        truncation=True
    )

    source = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True
    )

    sources.append(source)

# Calculate metrics using our reusable module
rouge_scores = compute_rouge(predictions, references)

overlap_scores = compute_source_overlap(
    sources,
    predictions
)

print("ROUGE:", rouge_scores)
print("\nSource overlap:", overlap_scores)

ABOVE there, What are we checking?

We're checking whether the functions in metrics.py reproduce the evaluation results we previously calculated directly in the notebook.

Expected ROUGE results should be approximately:

Metric	Previous result

ROUGE-1	0.4674

ROUGE-2	0.2629

ROUGE-L	0.3522

ROUGE-Lsum	0.4082

Small differences may arise from implementation or rounding, but the results should match if the same inputs and settings are used

FINISH E0- SAVING FINAL TEACHERS METRICS

In [ ]:
import json
import os
baseline_metrics = {
    "experiment": "E0_teacher",
    "num_examples": len(teacher_results),

    "rouge": {
        key: float(value)
        for key, value in rouge_scores.items()
    },

    "source_overlap": overlap_scores
}

RESULTS_DIR = (
    "/content/drive/MyDrive/research_artifacts/"
    "mini_bart/teacher_baseline"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

output_path = os.path.join(
    RESULTS_DIR,
    "teacher_metrics.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=4)

print("Teacher metrics saved!")
print(output_path)

VERIFY

In [ ]:
#verifying
print(os.listdir(RESULTS_DIR))

E0 EVALUATION COMPLETED. YAAAAYYYYY🥳🥳

E1 starting here

making sure of libraries once again

In [ ]:
!pip install -q datasets transformers accelerate

import our functions

uploading my local data.py  to colab

verifying GPU again

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected.")

verifying libraries

In [ ]:
%pip install evaluate

In [ ]:

import torch
import transformers
import datasets
import accelerate
import evaluate

print("Environment verification")
print("-" * 40)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Evaluate:", evaluate.__version__)

# Actual GPU computation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(2048, 2048, device=device)
y = torch.randn(2048, 2048, device=device)
z = x @ y

torch.cuda.synchronize()

print("\nGPU:", torch.cuda.get_device_name(0))
print("Test tensor device:", z.device)
print("Test computation successful:", torch.isfinite(z).all().item())
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GB"
)


connecting drive(got confused and messed up)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Mini-BART"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "logs"), exist_ok=True)

print("Project storage ready:", PROJECT_DIR)
print("Checkpoints:", os.path.join(PROJECT_DIR, "checkpoints"))
print("Logs:", os.path.join(PROJECT_DIR, "logs"))


checking location of project in drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart"

print("Project exists:", os.path.exists(PROJECT_DIR))
print("\nProject contents:")

for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")

    if level >= 2:
        dirs[:] = []
        continue

    for file in files:
        print("  " * (level + 1) + file)

checking if train.py and model.py are there in drive if not then its not syncrhonised with local and github


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Research/projects/01-mini-bart"

os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())

print("\n--- train.py ---")
with open("src/train.py", "r", encoding="utf-8") as f:
    print(f.read())

print("\n--- model.py ---")
with open("src/model.py", "r", encoding="utf-8") as f:
    print(f.read())

synchronizing Drive

In [ ]:
%cd /content/drive/MyDrive/Research/projects/01-mini-bart

!git pull origin main

the above failed. so nvm it

In [ ]:
import shutil

source = "projects/01-mini-bart/src/data.py"
backup = "projects/01-mini-bart/src/data_drive_backup.py"

shutil.copy2(source, backup)

print("Backup created:", backup)

In [ ]:
%cd /content/drive/MyDrive/Research

!git status
!git stash push -m "Preserve Drive data.py before GitHub sync" -- projects/01-mini-bart/src/data.py

In [ ]:
!git pull --ff-only origin main

In [ ]:
!git status

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Research/projects/01-mini-bart")

for filename in ["src/model.py", "src/train.py", "src/data.py"]:
    path = project / filename
    print(f"\n{filename}")
    print("Exists:", path.exists())
    print("Size:", path.stat().st_size if path.exists() else "Missing")
    print(path.read_text()[:200] if path.exists() else "")

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Research/projects/01-mini-bart"
)

sys.path.insert(0, str(PROJECT_DIR))

from src.model import (
    build_student_model,
    initialize_student_from_teacher,
)

from src.train import (
    create_optimizer_and_scheduler,
    train_one_epoch,
)

print("All imports successful!")

print("\nModel functions:")
print(build_student_model)
print(initialize_student_from_teacher)

print("\nTraining functions:")
print(create_optimizer_and_scheduler)
print(train_one_epoch)

inspecting synchronized env

In [ ]:
import os
import subprocess

RESEARCH_DIR = "/content/drive/MyDrive/Research"
PROJECT_DIR = os.path.join(RESEARCH_DIR, "projects/01-mini-bart")

os.chdir(RESEARCH_DIR)

print("Project directory:", PROJECT_DIR)
print("\nLatest Git commit:")
subprocess.run(["git", "log", "-1", "--oneline"])

print("\nGit status:")
subprocess.run(["git", "status", "--short"])

print("\nRequired files:")
files = [
    "src/data.py",
    "src/model.py",
    "src/train.py",
    "src/distillation.py",
    "src/evaluate.py",
    "src/metrics.py",
    "notebooks/00_setup.ipynb",
]

for file in files:
    path = os.path.join(PROJECT_DIR, file)
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'}: {file}")

print("\nTraining functions:")
with open(os.path.join(PROJECT_DIR, "src/train.py")) as f:
    for line in f:
        if line.startswith("def ") or line.startswith("class "):
            print(line.strip())

print("\nData functions:")
with open(os.path.join(PROJECT_DIR, "src/data.py")) as f:
    for line in f:
        if line.startswith("def ") or line.startswith("class "):
            print(line.strip())

inspecting source code 1)train.py

In [ ]:
from pathlib import Path

train_file = Path(PROJECT_DIR) / "src/train.py"

print("=" * 70)
print("CURRENT src/train.py")
print("=" * 70)

print(train_file.read_text())

inspecting data.py

In [ ]:
data_file = Path(PROJECT_DIR) / "src/data.py"

print("=" * 70)
print("CURRENT src/data.py")
print("=" * 70)

print(data_file.read_text())

DOWN IS HOW YOU PULL FROM GITHUB TO GOOGLE DRIVE!!

In [ ]:
import os
import subprocess

RESEARCH_DIR = "/content/drive/MyDrive/Research"

os.chdir(RESEARCH_DIR)

# Check repository status first
subprocess.run(["git", "status"], check=True)

# Pull latest committed code
result = subprocess.run(
    ["git", "pull", "origin", "main"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Git pull failed. Resolve this before continuing.")

In [ ]:
from pathlib import Path

train_path = Path(
    "/content/drive/MyDrive/Research/projects/01-mini-bart/src/train.py"
)

print("Exists:", train_path.exists())
print("Size:", train_path.stat().st_size, "bytes")

print("\nFunctions:")
for line in train_path.read_text().splitlines():
    if line.startswith("def "):
        print(line)

In [ ]:
import shutil

drive_path = "/content/drive/MyDrive"

total, used, free = shutil.disk_usage(drive_path)

print(f"Total: {total / 1024**3:.2f} GB")
print(f"Used: {used / 1024**3:.2f} GB")
print(f"Free: {free / 1024**3:.2f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Research/projects/01-mini-bart"
)

assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"

sys.path.insert(0, str(PROJECT_DIR))

TRAIN_FILE = PROJECT_DIR / "src" / "train.py"

assert TRAIN_FILE.exists(), f"train.py not found: {TRAIN_FILE}"

print("Project directory:", PROJECT_DIR)
print("Training file:", TRAIN_FILE)
print("\nRequired functions:")

source = TRAIN_FILE.read_text(encoding="utf-8")

for function in [
    "create_optimizer_and_scheduler",
    "train_one_epoch",
    "validate_one_epoch",
    "save_checkpoint",
    "train_model",
]:
    status = "FOUND" if f"def {function}" in source else "MISSING"
    print(f"{function}: {status}")

checking files, storage etc


In [ ]:
from pathlib import Path
import torch
import os

# 1. Check project files
SRC_DIR = PROJECT_DIR / "src"

required_files = [
    "data.py",
    "model.py",
    "train.py",
    "metrics.py",
    "evaluate.py",
]

print("PROJECT FILES")
for filename in required_files:
    path = SRC_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

# 2. Check likely teacher checkpoint locations
teacher_candidates = [
    Path("/content/drive/MyDrive/research_artifacts/mini_bart/teacher"),
    Path("/content/drive/MyDrive/Research/artifacts/teacher"),
    Path("/content/drive/MyDrive/Research/teacher"),
]

print("\nTEACHER CHECKPOINT")
for path in teacher_candidates:
    print(f"\n{path}")
    if path.exists():
        print("FOUND")
        print("Contents:", [p.name for p in path.iterdir()][:15])
    else:
        print("Not found")

# 3. GPU status
print("\nGPU")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

# 4. Drive free space
stat = os.statvfs("/content/drive/MyDrive")
free_gb = stat.f_bavail * stat.f_frsize / (1024**3)
print("\nApprox. Drive free space:", round(free_gb, 2), "GB")

In [ ]:
import gc
import torch
from transformers import BartForConditionalGeneration, AutoTokenizer

from src.model import (
    build_student_model,
    initialize_student_from_teacher,
)

# Paths
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
device = torch.device("cuda")

# 1. Load the pretrained teacher on CPU
print("Loading teacher...")
teacher = BartForConditionalGeneration.from_pretrained(TEACHER_DIR)
teacher.eval()

# 2. Build the 6-encoder / 6-decoder student
print("Building Mini-BART...")
student = build_student_model(teacher)

# 3. Transfer selected teacher weights into the student
print("Transferring teacher weights...")
initialize_student_from_teacher(teacher, student)

# 4. Load tokenizer from the same local checkpoint
tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)

# 5. Release teacher from memory — E1 does not need it during training
del teacher
gc.collect()

# 6. Move student to the T4
student = student.to(device)
student.train()

# 7. Report architecture and parameter counts
student_params = sum(p.numel() for p in student.parameters())

print("\n========== MINI-BART STATUS ==========")
print("Device:", next(student.parameters()).device)
print("Encoder layers:", student.config.encoder_layers)
print("Decoder layers:", student.config.decoder_layers)
print(f"Student parameters: {student_params:,}")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("Tokenizer vocabulary:", len(tokenizer))
print("======================================")

loading dataset

In [ ]:
import importlib
import src.data

importlib.reload(src.data)

from src.data import load_cnn_dailymail

raw_dataset = load_cnn_dailymail()

print("\nDataset loaded successfully!")
for split, data in raw_dataset.items():
    print(f"{split}: {len(data):,} examples")

taking some examples


In [ ]:
import os
import torch
from src.data import (
    load_cnn_dailymail,
    prepare_dataset,
    create_dataloaders,
)

# Keep temporary dataset cache on Colab storage, not Google Drive
os.environ["HF_DATASETS_CACHE"] = "/content/hf_datasets_cache"

print("Loading CNN/DailyMail v3.0.0...")
raw_dataset = load_cnn_dailymail()

print("\nOriginal dataset sizes:")
for split, data in raw_dataset.items():
    print(f"{split}: {len(data):,} examples")

print("\nPreparing benchmark subset...")
tokenized_dataset = prepare_dataset(
    raw_dataset,
    tokenizer,
    max_train_samples=128,
    max_val_samples=32,
    max_test_samples=0,
)

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=2,
    num_workers=0,
)

print("\n========== DATASET BENCHMARK ==========")
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))
print("Test examples:", len(tokenized_dataset["test"]))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: {tuple(value.shape)}")

print("=======================================")

This cell performs forward propagation, cross-entropy loss calculation, backward propagation, gradient clipping, and one optimizer update.used FP16 mixed precision, matching our planned training configuration

In [ ]:
import time
import torch
from torch.optim import AdamW

# Prepare model
student.train()
student.zero_grad(set_to_none=True)

# One batch
batch = next(iter(train_loader))
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

# Optimizer for benchmark only
optimizer = AdamW(
    student.parameters(),
    lr=5e-5,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler("cuda")

# Reset GPU memory statistics
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Benchmark
torch.cuda.synchronize()
start_time = time.time()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    outputs = student(**batch)
    loss = outputs.loss

scaler.scale(loss).backward()

scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(
    student.parameters(),
    max_norm=1.0,
)

scaler.step(optimizer)
scaler.update()

torch.cuda.synchronize()
elapsed = time.time() - start_time

# Results
peak_memory = torch.cuda.max_memory_allocated() / 1024**3
reserved_memory = torch.cuda.max_memory_reserved() / 1024**3

print("\n========== GPU TRAINING BENCHMARK ==========")
print(f"Loss: {loss.item():.4f}")
print(f"Step time: {elapsed:.2f} seconds")
print(f"Peak allocated GPU memory: {peak_memory:.2f} GB")
print(f"Peak reserved GPU memory: {reserved_memory:.2f} GB")
print(f"Remaining GPU memory: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")
print("Optimizer update: completed")
print("============================================")

In [ ]:
import time
import gc
import torch
from torch.optim import AdamW
from src.data import create_dataloaders

for batch_size in [4, 8]:

    print(f"\nTesting batch size: {batch_size}")

    gc.collect()
    torch.cuda.empty_cache()
    student.zero_grad(set_to_none=True)

    try:
        train_loader_test, _, _ = create_dataloaders(
            tokenized_dataset,
            tokenizer,
            batch_size=batch_size,
            num_workers=0,
        )

        batch = next(iter(train_loader_test))
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer_test = AdamW(
            student.parameters(),
            lr=5e-5,
            weight_decay=0.01,
        )

        scaler_test = torch.amp.GradScaler("cuda")

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        start = time.time()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            outputs = student(**batch)
            loss = outputs.loss

        scaler_test.scale(loss).backward()
        scaler_test.unscale_(optimizer_test)

        torch.nn.utils.clip_grad_norm_(
            student.parameters(), 1.0
        )

        scaler_test.step(optimizer_test)
        scaler_test.update()

        torch.cuda.synchronize()
        elapsed = time.time() - start

        print(f"Loss: {loss.item():.4f}")
        print(f"Step time: {elapsed:.2f} seconds")
        print(
            f"Peak GPU memory: "
            f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
        )
        print(f"Time per example: {elapsed / batch_size:.3f} seconds")

        del optimizer_test, scaler_test, batch
        del train_loader_test

    except torch.cuda.OutOfMemoryError:
        print("OUT OF MEMORY — this batch size is not viable.")
        student.zero_grad(set_to_none=True)
        gc.collect()
        torch.cuda.empty_cache()

print("\nBatch-size benchmark complete.")

In [ ]:
import importlib
import src.train

# Reload the latest version from Google Drive
importlib.reload(src.train)

print("Available functions:")
for name in [
    "create_optimizer_and_scheduler",
    "train_one_epoch",
    "validate_one_epoch",
    "save_checkpoint",
    "train_model",
]:
    print(f"{name}: {hasattr(src.train, name)}")

In [ ]:
import shutil
import inspect
from pathlib import Path

from src.train import save_checkpoint

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return "Not found"

    total = sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    )
    return f"{total / 1024**3:.2f} GB"

print("========== STORAGE ==========")

print("Drive:")
!df -h /content/drive

print("\nTeacher checkpoint size:")
print(folder_size_gb(
    "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
))

print("\nProject checkpoint folder:")
print(folder_size_gb(PROJECT_DIR / "checkpoints"))

print("\n========== CHECKPOINT FUNCTION ==========")
print(inspect.getsource(save_checkpoint))

In [ ]:
import inspect
from pathlib import Path
from src.train import train_model, create_optimizer_and_scheduler

print("========= TRAINING FUNCTION =========")
print(inspect.signature(train_model))
print(inspect.getsource(train_model))

print("\n========= DATALOADERS =========")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\n========= BATCH CONFIGURATION =========")
batch = next(iter(train_loader))
for key, value in batch.items():
    print(f"{key}: {tuple(value.shape)}")

print("\n========= MODEL =========")
print("Device:", next(student.parameters()).device)
print("Parameters:", sum(p.numel() for p in student.parameters()))

print("\n========= CHECKPOINT DIRECTORY =========")
OUTPUT_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/student_baseline"
print("Output:", OUTPUT_DIR)
print("Exists:", Path(OUTPUT_DIR).exists())

In [ ]:
import inspect
from src.data import prepare_dataset, create_dataloaders

print("========= RAW DATASET =========")
for split in raw_dataset:
    print(split, len(raw_dataset[split]))

print("\n========= PREPROCESSING SIGNATURE =========")
print(inspect.signature(prepare_dataset))

print("\n========= DATALOADER SIGNATURE =========")
print(inspect.signature(create_dataloaders))

print("\n========= PREPROCESSING IMPLEMENTATION =========")
print(inspect.getsource(prepare_dataset))

In [ ]:
import importlib
import inspect
from pathlib import Path
import src.train

# Refresh the module to ensure we're using the latest source
importlib.reload(src.train)

save_checkpoint = src.train.save_checkpoint

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return "Not found"

    total = sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    )
    return f"{total / 1024**3:.2f} GB"

print("========== STORAGE ==========")

print("Drive:")
!df -h /content/drive

print("\nColab local storage:")
!df -h /content

print("\nTeacher checkpoint size:")
print(folder_size_gb(
    "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
))

print("\nProject checkpoint folder:")
print(folder_size_gb(Path(PROJECT_DIR) / "checkpoints"))

print("\nProject artifacts folder:")
print(folder_size_gb(Path(PROJECT_DIR) / "artifacts"))

print("\n========== CHECKPOINT FUNCTION ==========")
print(inspect.getsource(save_checkpoint))

In [ ]:
import inspect
from src.train import train_model

print(inspect.getsource(train_model))

# **FULL DATASET PREPROCESSING**

In [ ]:

import os
import gc
import torch
from pathlib import Path

# Keep dataset cache on Colab's local disk, NOT Google Drive
os.environ["HF_DATASETS_CACHE"] = "/content/hf_datasets_cache"

CACHE_DIR = Path("/content/hf_datasets_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Available local disk space:")
!df -h /content

# Load CNN/DailyMail 3.0.0
print("\nLoading CNN/DailyMail...")
raw_dataset = load_cnn_dailymail()

print("\nOriginal dataset sizes:")
for split in raw_dataset:
    print(f"{split}: {len(raw_dataset[split]):,}")

# Tokenize ALL training and validation examples.
# Explicitly select zero test examples to preserve the test set.
print("\nTokenizing full training and validation datasets...")

tokenized_dataset = prepare_dataset(
    raw_dataset,
    tokenizer,
    max_train_samples=None,
    max_val_samples=None,
    max_test_samples=0,
)

print("\nTokenized dataset sizes:")
for split in tokenized_dataset:
    print(f"{split}: {len(tokenized_dataset[split]):,}")

# Create dataloaders with our selected batch size
print("\nCreating dataloaders...")

train_loader, val_loader, test_loader = create_dataloaders(
    tokenized_dataset,
    tokenizer,
    batch_size=8,
    num_workers=0,
)

print("\n========== DATASET READY ==========")
print(f"Training examples: {len(train_loader.dataset):,}")
print(f"Validation examples: {len(val_loader.dataset):,}")
print(f"Test examples: {len(test_loader.dataset):,}")
print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

# Inspect one batch
batch = next(iter(train_loader))

print("\nSample batch shapes:")
for key, value in batch.items():
    print(f"{key}: {tuple(value.shape)}")

del batch
gc.collect()

print("\nLocal disk after preprocessing:")
!df -h /content


# **Reinitialize Mini-BART and test a full-length batch**

In [ ]:

import gc
import time
import torch
from torch.optim import AdamW
from transformers import BartForConditionalGeneration

# Release the benchmark student
del student
gc.collect()
torch.cuda.empty_cache()

# Reinitialize from the original teacher
print("Loading teacher...")
teacher = BartForConditionalGeneration.from_pretrained(TEACHER_DIR)
teacher.eval()

print("Building and initializing Mini-BART...")
student = build_student_model(teacher)
initialize_student_from_teacher(teacher, student)

del teacher
gc.collect()

student = student.to(device)
student.train()

print("Mini-BART reinitialized successfully.")

# Test one actual full-length batch
batch = next(iter(train_loader))
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

optimizer_test = AdamW(
    student.parameters(),
    lr=5e-5,
    weight_decay=0.01,
)

scaler_test = torch.amp.GradScaler("cuda")

student.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.time()

with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = student(**batch)
    loss = outputs.loss

scaler_test.scale(loss).backward()
scaler_test.unscale_(optimizer_test)

torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)

scaler_test.step(optimizer_test)
scaler_test.update()

torch.cuda.synchronize()
elapsed = time.time() - start

print("\n========== FULL-LENGTH BATCH TEST ==========")
print(f"Input shape: {tuple(batch['input_ids'].shape)}")
print(f"Label shape: {tuple(batch['labels'].shape)}")
print(f"Loss: {loss.item():.4f}")
print(f"Step time: {elapsed:.2f} seconds")
print(
    f"Peak GPU memory: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

# Clean up temporary test optimizer and gradients
del optimizer_test, scaler_test, outputs, batch
student.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

print("\nFull-length batch test completed.")


# **Launch E1 training**

In [ ]:

import gc
import torch
from transformers import BartForConditionalGeneration
from src.train import train_model

# Release the previous student
del student
gc.collect()
torch.cuda.empty_cache()

# Restore original teacher
print("Loading teacher...")
teacher = BartForConditionalGeneration.from_pretrained(TEACHER_DIR)
teacher.eval()

# Rebuild student with original teacher weights
print("Initializing Mini-BART...")
student = build_student_model(teacher)
initialize_student_from_teacher(teacher, student)

del teacher
gc.collect()
torch.cuda.empty_cache()

student = student.to(device)
student.train()

# Verify training configuration
BATCH_SIZE = 8
NUM_EPOCHS_E1 = 3

OUTPUT_DIR = (
    "/content/drive/MyDrive/research_artifacts/mini_bart/E1"
)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n========== E1 CONFIGURATION ==========")
print(f"Training examples: {len(train_loader.dataset):,}")
print(f"Validation examples: {len(val_loader.dataset):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS_E1}")
print(f"Learning rate: 5e-5")
print(f"Objective: Supervised cross-entropy only")
print(f"Checkpoint directory: {OUTPUT_DIR}")

# Launch training
history = train_model(
    model=student,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    device=device,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS_E1,
)

print("\n========== E1 TRAINING FINISHED ==========")
print(history)
